# 05 - Multi-Factor Alpha Combination & Sector-Neutral Portfolio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/05_multi_factor_sector_neutral_portfolio.ipynb)

In this notebook, we run the **institutional multi-factor workflow** end to end with **polars-anta**:
three orthogonal alpha signals (momentum, mean reversion and liquidity) are computed for a
100-ticker panel, cleaned cross-sectionally (MAD winsorization and z-score per date),
neutralized against sector exposure, combined into one equal-weighted composite alpha, and
then compared factor-by-factor with Rank IC / ICIR and dollar-neutral backtests.

> The data is synthetic and the backtest ignores borrow costs, so the results are illustrative only.

In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta matplotlib

In [ ]:
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta as pa

print(f"Polars Version      : {pl.__version__}")
print(f"polars-anta Version : {pa.__version__}")

## 1. Simulate a 100-Ticker, 5-Sector Panel

500 trading days for 100 tickers spread evenly across 5 GICS-style sectors. Each ticker's
return mixes a sector-wide component, a slow latent alpha component (yesterday's latent
state leaks into today's return, so price-based factors have something real to find) and
idiosyncratic noise. Everything is generated locally from a fixed seed - no network data.

In [ ]:
rng = np.random.default_rng(42)
n_tickers, n_days = 100, 500
sectors = ["Technology", "Financials", "Health Care", "Energy", "Industrials"]
dates = [datetime(2024, 1, 2) + timedelta(days=int(i)) for i in range(n_days)]

# One sector-wide return path per sector.
sector_ret = {s: rng.normal(0.0002, 0.008, n_days) for s in sectors}

frames = []
for idx in range(n_tickers):
    sector = sectors[idx % len(sectors)]
    latent = rng.normal(0.0, 1.0, n_days)
    ret = (
        0.0003
        + sector_ret[sector]
        + 0.0045 * np.roll(latent, 1)
        + rng.normal(0.0, 0.013, n_days)
    )
    ret[0] = 0.0
    close = (40.0 + idx) * np.cumprod(1.0 + ret)
    open_ = close * (1.0 - 0.0015 * latent)
    frames.append(
        pl.DataFrame(
            {
                "date": dates,
                "ticker": [f"TICK_{idx + 1:03d}"] * n_days,
                "sector": [sector] * n_days,
                "open": open_,
                "high": np.maximum(open_, close) * 1.005,
                "low": np.minimum(open_, close) * 0.995,
                "close": close,
                "volume": rng.lognormal(13.5, 0.4, n_days),
            }
        )
    )

panel = pl.concat(frames).sort(["ticker", "date"])
print(panel.head())

## 2. Compute Three Orthogonal Alpha Signals

One factor per alpha family, each computed per ticker with `.over("ticker")`:

- **Momentum**: RSI(14) via `.ta.rsi()` - recent winners keep winning.
- **Mean reversion**: WorldQuant `alpha_001` via `.anta.alpha_001()` - fades stretched moves.
- **Liquidity**: Amihud illiquidity via `.anta.amihud_illiquidity()` - the price impact per unit of dollar volume (the illiquidity premium).

In [ ]:
factors = panel.with_columns(
    pl.col("close").ta.rsi(14).over("ticker").alias("momentum"),
    pl.col("close").anta.alpha_001(over="ticker").alias("mean_reversion"),
    pl.col("close").anta.amihud_illiquidity("volume", period=20).over("ticker").alias("liquidity"),
)
print(factors.select(["date", "ticker", "sector", "momentum", "mean_reversion", "liquidity"]).tail(5))

## 3. Clean Cross-Sectionally and Neutralize Sector Exposure

Raw factors are not comparable: different scales, fat tails, and a sector tilt (e.g. every
Energy name scoring high together). The institutional cleaning pipeline, applied per date:

1. **Winsorize** outliers with `.anta.cs_winsorize(over="date", method="mad")`.
2. **Z-score** with `.anta.cs_zscore(over="date")` so factors share one scale.
3. **Sector-neutralize** with `.anta.neutralize(by="sector", over="date")`, which removes each sector's cross-sectional mean so the book carries no net sector bet.

The **composite alpha** is the equal-weighted average of the three cleaned factors.

In [ ]:
def clean(expr: pl.Expr) -> pl.Expr:
    return (
        expr.anta.cs_winsorize(over="date", method="mad")
        .anta.cs_zscore(over="date")
        .anta.neutralize(by="sector", over="date")
    )


cleaned = factors.with_columns(
    clean(pl.col("momentum")).alias("momentum_c"),
    clean(pl.col("mean_reversion")).alias("mean_reversion_c"),
    clean(pl.col("liquidity")).alias("liquidity_c"),
).with_columns(
    ((pl.col("momentum_c") + pl.col("mean_reversion_c") + pl.col("liquidity_c")) / 3.0).alias(
        "composite"
    )
)

# Warm-up windows (RSI, Amihud) and the sector-neutral means leave some nulls - drop them
# once so every comparison below uses the exact same rows.
cleaned = cleaned.drop_nulls(
    subset=["momentum_c", "mean_reversion_c", "liquidity_c", "composite"]
)
print(f"Cleaned panel: {cleaned.height} rows")
print(
    cleaned.group_by("sector").agg(
        pl.col("composite").mean().round(4).alias("composite_mean_by_sector")
    )
)

## 4. Compare Factors with Rank IC and ICIR

`ic_summary` scores each factor against next-day forward returns across every date's
cross-section. A factor that only works because the others carry it shows up here before
any backtest flatters it.

In [ ]:
eval_df = cleaned.anta.forward_returns(periods=(1,), group_by="ticker").drop_nulls(
    subset=["fwd_ret_1"]
)

factor_cols = ["momentum_c", "mean_reversion_c", "liquidity_c", "composite"]
ic_rows = []
for col in factor_cols:
    stats = eval_df.anta.ic_summary(col, "fwd_ret_1", group_by="date", method="spearman")
    ic_rows.append(
        {
            "factor": col,
            "ic_mean": round(stats["ic_mean"], 4),
            "ic_std": round(stats["ic_std"], 4),
            "icir": round(stats["icir"], 3),
            "hit_rate": round(stats["hit_rate"], 3),
            "n_periods": int(stats["n_periods"]),
        }
    )
print(pl.DataFrame(ic_rows))

## 5. Dollar-Neutral Backtests: Individual Factors vs Composite

Each cleaned factor (and the composite) drives the same dollar-neutral long-short book
through `df.anta.backtest(...)`; `pa.compare_strategies` lines the tearsheets up side by side.

In [ ]:
results = {
    name: cleaned.anta.backtest(
        signal=col,
        group_by="ticker",
        time_col="date",
        weight_scheme="dollar_neutral",
        fee_rate=0.0001,
        slippage_rate=0.0001,
        initial_capital=1_000_000.0,
    )
    for name, col in [
        ("Momentum", "momentum_c"),
        ("Mean Reversion", "mean_reversion_c"),
        ("Liquidity", "liquidity_c"),
        ("Composite", "composite"),
    ]
}

comparison = pa.compare_strategies(results).select(
    [
        "strategy",
        (pl.col("total_return") * 100).round(2).alias("total_ret_%"),
        (pl.col("sharpe_ratio")).round(2),
        (pl.col("sortino_ratio")).round(2),
        (pl.col("max_drawdown") * 100).round(2).alias("max_dd_%"),
        pl.col("n_trades").cast(pl.Int64),
    ]
)
print(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, res in results.items():
    ax.plot(res.data["date"], res.data["equity_curve"], label=name, lw=1.4)
ax.set_title("Dollar-Neutral Equity Curves: Factors vs Composite", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Equity ($)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Next Steps

- Weight the composite by rolling ICIR instead of equal weights, and re-check the IC table.
- Add a second neutralization layer (e.g. market-cap or volatility buckets) with `neutralize(by=[...], over="date")`.
- Stress the winner: raise `fee_rate`/`slippage_rate`, and inspect turnover before trusting the composite's edge.